# 31 - External-Mapped Fine-Tuned RAG Generation Evaluation

Runs only the new full fine-tuned stack: external-mapped tuned embedding + external-mapped tuned reranker + external-mapped QLoRA LLM. Existing base generation metrics are not recomputed.

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes peft "sentence-transformers>=5.1.0" faiss-cpu rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/TURKISH_LEGAL_RAG')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))
%cd {DRIVE_ROOT}

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Project:', DRIVE_ROOT)

In [ ]:
from src.generation import run_rag_generation
from src.evaluation_qa import evaluate_generation_predictions

benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b_external_mapped_lora_v1'
reranker_model = DRIVE_ROOT / 'models/reranker_tuned/qwen3_reranker_8b_external_mapped_lora_v1'
adapter_path = DRIVE_ROOT / 'models/adapters/qwen3_32b_external_mapped_qlora_v1'
output_dir = DRIVE_ROOT / 'outputs/external_mapped_generation'
output_dir.mkdir(parents=True, exist_ok=True)

for required in [benchmark_csv, index_root / 'index_manifest.json', reranker_model, adapter_path]:
    if not Path(required).exists():
        raise FileNotFoundError(required)

print('Benchmark:', benchmark_csv)
print('Index:', index_root)
print('Reranker:', reranker_model)
print('Adapter:', adapter_path)

In [ ]:
# Set limit=20 for a quick smoke run. Use limit=None for the final full 190-question run.
limit = None
predictions_csv = output_dir / 'external_mapped_qlora_predictions.csv'

run_config = run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=predictions_csv,
    output_run_config_json=output_dir / 'external_mapped_qlora_run_config.json',
    llm_model='Qwen/Qwen3-32B',
    retriever_mode='dense',
    top_k_context=10,
    candidate_k=30,
    device=device,
    max_new_tokens=384,
    temperature=0.0,
    top_p=1.0,
    input_max_length=8192,
    limit=limit,
    load_in_4bit=True,
    adapter_path=adapter_path,
    system_name='external_mapped_full_tuned_stack',
    reranker_model=str(reranker_model),
    reranker_batch_size=4,
)
print(json.dumps(run_config, ensure_ascii=False, indent=2))

In [ ]:
summary = evaluate_generation_predictions(
    predictions_csv=predictions_csv,
    output_eval_csv=output_dir / 'external_mapped_qlora_eval.csv',
    output_summary_json=output_dir / 'external_mapped_qlora_summary.json',
)
print(json.dumps(summary['metrics'], ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd

eval_df = pd.read_csv(output_dir / 'external_mapped_qlora_eval.csv', dtype=str, keep_default_na=False)
preview_cols = ['question_id', 'question', 'generated_answer', 'retrieved_citations', 'citation_present', 'citation_gold_match', 'grounded_citation_score']
display(eval_df[preview_cols].head(5))